# Frontier Model Benchmark — UHCS Microstructure

Benchmark GPT-4.1 and GPT-5 on 6-class UHCS microconstituent classification.

**Models:**
- GPT-4.1 (ether-openai, t=0.7)
- GPT-5 (ether-project-resource, t=1 — reasoning model, locked)

**Test set:** ~120 images (20% stratified from 598 labeled)

## Setup

In [ ]:
import os, json, re, time, base64
import numpy as np
from PIL import Image
from collections import defaultdict, Counter
import matplotlib.pyplot as plt

# !pip install openai -q

## Config

In [ ]:
from openai import AzureOpenAI
from config import CLASSES, REF_IMAGE_PATH, TEST_MANIFEST
from config import make_prompt_zs, make_prompt_fs_ref, make_prompt_fs_query

# --- GPT-4.1 ---
GPT41_ENDPOINT = 'https://ether-openai.openai.azure.com/'
GPT41_DEPLOYMENT = 'gpt-4.1'
GPT41_API_VERSION = '2024-12-01-preview'
GPT41_API_KEY = os.environ.get('AZURE_OPENAI_API_KEY_41', '')

# --- GPT-5 ---
GPT5_ENDPOINT = 'https://ether-project-resource.openai.azure.com/'
GPT5_DEPLOYMENT = 'gpt-5'
GPT5_API_VERSION = '2024-12-01-preview'
GPT5_API_KEY = os.environ.get('AZURE_OPENAI_API_KEY_5', '')

client_41 = AzureOpenAI(azure_endpoint=GPT41_ENDPOINT, api_key=GPT41_API_KEY, api_version=GPT41_API_VERSION)
client_5 = AzureOpenAI(azure_endpoint=GPT5_ENDPOINT, api_key=GPT5_API_KEY, api_version=GPT5_API_VERSION)

print(f'GPT-4.1: key set: {bool(GPT41_API_KEY)}')
print(f'GPT-5:   key set: {bool(GPT5_API_KEY)}')

## Load Data

In [ ]:
with open(TEST_MANIFEST) as f:
    manifest = json.load(f)
print(f'Test images: {len(manifest)}')
cls_counts = Counter(e['class'] for e in manifest)
for cls in CLASSES:
    print(f'  {cls}: {cls_counts.get(cls, 0)}')

# Load reference image
ref_b64 = None
if os.path.exists(REF_IMAGE_PATH):
    with open(REF_IMAGE_PATH, 'rb') as f:
        ref_b64 = base64.b64encode(f.read()).decode('utf-8')
    print(f'\nReference image loaded: {REF_IMAGE_PATH}')
else:
    print(f'WARNING: {REF_IMAGE_PATH} not found')

## Helpers

In [ ]:
def encode_image(path):
    with open(path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')

def call_api(client, deployment, image_b64, prompt, ref_b64=None, ref_prompt=None, temperature=None):
    content = []
    if ref_b64 and ref_prompt:
        content.append({'type': 'image_url', 'image_url': {'url': f'data:image/png;base64,{ref_b64}'}})
        content.append({'type': 'text', 'text': ref_prompt})
    content.append({'type': 'image_url', 'image_url': {'url': f'data:image/png;base64,{image_b64}'}})
    content.append({'type': 'text', 'text': prompt})
    kwargs = dict(model=deployment, max_completion_tokens=8192,
                  messages=[{'role': 'user', 'content': content}])
    if temperature is not None:
        kwargs['temperature'] = temperature
    resp = client.chat.completions.create(**kwargs)
    return resp.choices[0].message.content

def parse_response(raw):
    if not raw: return None
    raw = raw.replace('<','').replace('>','')
    raw = re.sub(r'```json\s*', '', raw)
    raw = re.sub(r'```\s*', '', raw).strip()
    try:
        obj = json.loads(raw)
        if isinstance(obj, dict): return obj
    except: pass
    m = re.search(r'\{.*\}', raw, re.DOTALL)
    if m:
        try: return json.loads(m.group())
        except: pass
    dm = re.search(r'"primary_microconstituent"\s*:\s*"([\w+]+)"', raw)
    if dm: return {'primary_microconstituent': dm.group(1)}
    raw_lower = raw.lower()
    for cls in sorted(CLASSES, key=len, reverse=True):
        if cls in raw_lower: return {'primary_microconstituent': cls}
    return None

print('Helpers ready.')

## Benchmark Function

In [ ]:
def run_frontier_benchmark(manifest, client, deployment, mode, temperature=None, ref_b64=None, limit=None):
    data = manifest[:limit] if limit else manifest
    results = []; correct = 0; valid = 0; total_time = 0
    for i, entry in enumerate(data):
        img_path = entry['image']
        if not os.path.exists(img_path):
            print(f'  SKIP: {img_path}'); continue
        img_b64 = encode_image(img_path)
        mag = entry.get('magnification', 'unknown')
        gt = entry['class']
        t = time.time()
        try:
            if mode == 'few-shot' and ref_b64:
                raw = call_api(client, deployment, img_b64, make_prompt_fs_query(mag),
                               ref_b64=ref_b64, ref_prompt=make_prompt_fs_ref(), temperature=temperature)
            else:
                raw = call_api(client, deployment, img_b64, make_prompt_zs(mag), temperature=temperature)
        except Exception as e:
            raw = f'ERROR: {e}'
        elapsed = time.time() - t
        total_time += elapsed
        parsed = parse_response(raw)
        ok = False
        if parsed:
            valid += 1
            pred = parsed.get('primary_microconstituent', '').lower().strip()
            if pred == gt: ok = True; correct += 1
        results.append({'image': img_path, 'class': gt, 'magnification': mag,
            'predicted': parsed, 'raw': raw, 'correct': ok,
            'valid_json': parsed is not None, 'time_s': round(elapsed, 2)})
        if (i+1) % 20 == 0:
            n = i+1
            print(f'  [{n}/{len(data)}] Acc: {correct}/{n} ({correct/n*100:.0f}%) | JSON: {valid}/{n}')
        time.sleep(0.25)
    return results, correct, valid, total_time

print('Benchmark function ready.')

## Quick Test (GPT-4.1 ZS, 1 per class)

In [ ]:
quick = []
for cls in CLASSES:
    entry = next((e for e in manifest if e['class'] == cls), None)
    if entry: quick.append(entry)

print('Quick test — GPT-4.1 zero-shot')
print('=' * 60)
for entry in quick:
    img_b64 = encode_image(entry['image'])
    mag = entry.get('magnification', 'unknown')
    try:
        raw = call_api(client_41, GPT41_DEPLOYMENT, img_b64, make_prompt_zs(mag), temperature=0.7)
    except Exception as e:
        print(f'  {entry["class"]:>28} → ERROR: {e}'); continue
    parsed = parse_response(raw)
    gt = entry['class']
    pred = parsed.get('primary_microconstituent', '?') if parsed else '?'
    ok = '✓' if pred == gt else '✗'
    print(f'  {gt:>28} → {pred:<28} {ok}')
    time.sleep(0.3)

---
## GPT-4.1 Full Benchmark

In [ ]:
print('Running GPT-4.1 zero-shot...')
gpt41_zs_results, gpt41_zs_correct, gpt41_zs_valid, gpt41_zs_time = \
    run_frontier_benchmark(manifest, client_41, GPT41_DEPLOYMENT, 'zero-shot', temperature=0.7)
n = len(gpt41_zs_results)
print(f'\nGPT-4.1 ZS: Acc={gpt41_zs_correct}/{n} ({gpt41_zs_correct/n*100:.1f}%)')

In [ ]:
print('Running GPT-4.1 few-shot...')
gpt41_fs_results, gpt41_fs_correct, gpt41_fs_valid, gpt41_fs_time = \
    run_frontier_benchmark(manifest, client_41, GPT41_DEPLOYMENT, 'few-shot', temperature=0.7, ref_b64=ref_b64)
n = len(gpt41_fs_results)
print(f'\nGPT-4.1 FS: Acc={gpt41_fs_correct}/{n} ({gpt41_fs_correct/n*100:.1f}%)')

---
## GPT-5 Full Benchmark

In [ ]:
print('Running GPT-5 zero-shot...')
gpt5_zs_results, gpt5_zs_correct, gpt5_zs_valid, gpt5_zs_time = \
    run_frontier_benchmark(manifest, client_5, GPT5_DEPLOYMENT, 'zero-shot')
n = len(gpt5_zs_results)
print(f'\nGPT-5 ZS: Acc={gpt5_zs_correct}/{n} ({gpt5_zs_correct/n*100:.1f}%)')

In [ ]:
print('Running GPT-5 few-shot...')
gpt5_fs_results, gpt5_fs_correct, gpt5_fs_valid, gpt5_fs_time = \
    run_frontier_benchmark(manifest, client_5, GPT5_DEPLOYMENT, 'few-shot', ref_b64=ref_b64)
n = len(gpt5_fs_results)
print(f'\nGPT-5 FS: Acc={gpt5_fs_correct}/{n} ({gpt5_fs_correct/n*100:.1f}%)')

---
## Results Summary

In [ ]:
rows = []
for label, res, c, v, t in [
    ('GPT-4.1 (ZS, t=0.7)', gpt41_zs_results, gpt41_zs_correct, gpt41_zs_valid, gpt41_zs_time),
    ('GPT-4.1 (FS, t=0.7)', gpt41_fs_results, gpt41_fs_correct, gpt41_fs_valid, gpt41_fs_time),
    ('GPT-5 (ZS, t=1)',     gpt5_zs_results,  gpt5_zs_correct,  gpt5_zs_valid,  gpt5_zs_time),
    ('GPT-5 (FS, t=1)',     gpt5_fs_results,  gpt5_fs_correct,  gpt5_fs_valid,  gpt5_fs_time),
]:
    n = len(res)
    rows.append({'label': label, 'n': n, 'acc': round(c/n*100,1), 'json': round(v/n*100,1), 'time': round(t/n,2)})

# Load Qwen baselines if available
for mode in ['zero-shot', 'few-shot']:
    path = f'benchmark_results_{mode}.json'
    if os.path.exists(path):
        with open(path) as f:
            d = json.load(f)
        rows.insert(0, {'label': f'Qwen2.5-VL-3B ({mode})', 'n': d['total_images'],
            'acc': d['accuracy_pct'], 'json': d['json_validity_pct'], 'time': d['avg_inference_time_s']})

print(f'{"Method":<28} {"N":>5} {"Accuracy":>10} {"JSON":>7} {"Time/img":>10}')
print('=' * 62)
for r in rows:
    print(f'{r["label"]:<28} {r["n"]:>5} {r["acc"]:>9.1f}% {r["json"]:>6.0f}% {r["time"]:>9.1f}s')
print(f'{"Random chance (6 classes)":<28} {"":>5} {100/6:>9.1f}%')

## Per-Class Breakdown

In [ ]:
all_runs = [
    ('GPT-4.1 ZS', gpt41_zs_results),
    ('GPT-4.1 FS', gpt41_fs_results),
    ('GPT-5 ZS',   gpt5_zs_results),
    ('GPT-5 FS',   gpt5_fs_results),
]
for label, results in all_runs:
    print(f'\n{label}:')
    print(f'{"Class":<30} {"Correct":>10} {"Accuracy":>10}')
    print('-' * 52)
    by_class = defaultdict(list)
    for r in results: by_class[r['class']].append(r)
    for cls in CLASSES:
        cr = by_class[cls]
        c = sum(1 for r in cr if r['correct'])
        pct = c/len(cr)*100 if cr else 0
        print(f'{cls:<30} {c:>4}/{len(cr):<4} {pct:>9.1f}%')

## Save Results

In [ ]:
for label, results, correct, valid, tt, model_name in [
    ('gpt41_zs', gpt41_zs_results, gpt41_zs_correct, gpt41_zs_valid, gpt41_zs_time, 'gpt-4.1'),
    ('gpt41_fs', gpt41_fs_results, gpt41_fs_correct, gpt41_fs_valid, gpt41_fs_time, 'gpt-4.1'),
    ('gpt5_zs',  gpt5_zs_results,  gpt5_zs_correct,  gpt5_zs_valid,  gpt5_zs_time,  'gpt-5'),
    ('gpt5_fs',  gpt5_fs_results,  gpt5_fs_correct,  gpt5_fs_valid,  gpt5_fs_time,  'gpt-5'),
]:
    n = len(results)
    fname = f'benchmark_results_frontier_{label}.json'
    with open(fname, 'w') as f:
        json.dump({'model': model_name, 'mode': label, 'dataset': 'UHCS',
            'total_images': n, 'accuracy_pct': round(correct/n*100, 1),
            'json_validity_pct': round(valid/n*100, 1),
            'avg_inference_time_s': round(tt/n, 2),
            'results': results}, f, indent=2)
    print(f'Saved {fname}')